# Boosting

**Companion lesson:** https://ml-viz.vercel.app/courses/ensemble-methods/02-boosting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## AdaBoost: Sequential Boosting

Each learner focuses on the mistakes of the previous one.

In [ ]:
np.random.seed(42)
n = 80
X = np.sort(5 * np.random.rand(n))
y = np.sign(np.sin(X))  # +1 or -1

def fit_stump(X, y, weights):
    best_loss, best_t, best_p = np.inf, 0, 1
    for t in np.unique(X):
        for p in [1, -1]:
            pred = p * np.sign(X - t)
            pred[pred == 0] = 1
            loss = np.sum(weights * (pred != y))
            if loss < best_loss:
                best_loss, best_t, best_p = loss, t, p
    return best_t, best_p

# AdaBoost
weights = np.ones(n) / n
learners = []
errors = []

for m in range(10):
    t, p = fit_stump(X, y, weights)
    pred = p * np.sign(X - t)
    pred[pred == 0] = 1
    err = np.sum(weights * (pred != y))
    err = np.clip(err, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - err) / err)
    learners.append((t, p, alpha))
    errors.append(err)
    weights *= np.exp(-alpha * y * pred)
    weights /= weights.sum()

x_grid = np.linspace(0, 5, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show sequential focus
axes[0].scatter(X, y, c=y, cmap='RdYlBu', s=30, alpha=0.5)
for i in range(3):
    t, p, alpha = learners[i]
    pred = p * np.sign(x_grid - t)
    axes[0].plot(x_grid, pred * 0.5 - 1.5 - i * 0.3, color=['#f43f5e', '#eab308', '#14b8a6'][i],
                 linewidth=2, label=f'Round {i+1}')
axes[0].set_title('Each Round Adds a Stump', color='white')
axes[0].legend(fontsize=9)
axes[0].axis('off')

# Final ensemble prediction
final = np.zeros_like(x_grid)
for t, p, alpha in learners:
    final += alpha * (p * np.sign(x_grid - t))
axes[1].scatter(X, y, c='#818cf8', s=20, alpha=0.5)
axes[1].plot(x_grid, np.sign(final), color='#14b8a6', linewidth=2.5, label='Ensemble')
axes[1].plot(x_grid, np.sin(x_grid), '--', color='#94a3b8', label='True')
axes[1].set_title('AdaBoost Ensemble', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()